In [2]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "fyxical-poc"
DATASET    = "fyxical_poc"
client     = bigquery.Client(project=PROJECT_ID)

In [3]:
patients = client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET}.patient_intake`
""").to_dataframe()

print(f"Rows    : {len(patients)}")
print(f"Columns : {patients.shape[1]}")
print(patients.dtypes)

Rows    : 500
Columns : 26
patient_id                           object
age                                   Int64
sex                                  object
race_ethnicity                       object
occupation_type                      object
work_related_injury                  object
region_census                        object
division_census                      object
is_lower_back_pain                   object
pain_duration_weeks                 float64
pain_duration_category               object
pain_severity_score_0_10              Int64
pain_severity_category               object
osw_baseline_s1_pain_intensity        Int64
osw_baseline_s2_personal_care         Int64
osw_baseline_s3_lifting               Int64
osw_baseline_s4_walking               Int64
osw_baseline_s5_sitting               Int64
osw_baseline_s6_standing              Int64
osw_baseline_s7_sleeping              Int64
osw_baseline_s8_social_life           Int64
osw_baseline_s9_traveling             Int64
osw_b

In [4]:
exercises = client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET}.exercises`
""").to_dataframe()

print(f"Exercise records : {len(exercises)}")
print(f"Columns          : {exercises.columns.tolist()}")
print()
print(exercises.to_string(index=False))

Exercise records : 13
Columns          : ['pathway', 'exercise', 'sets', 'reps', 'frequency', 'description']

                        pathway                     exercise  sets  reps          frequency                                       description
                Urgent Referral Rest + MD Clearance Required     0     0 Pending assessment      No exercise until medical clearance obtained
Long-term Management + Referral      Graded Activity Program     1     1     Weekly with PT Structured activity exposure under PT supervision
          Home Exercise Program        Knee-to-Chest Stretch     2    30              Daily               Hip flexor and lumbar decompression
Long-term Management + Referral              Aquatic Therapy     2    15            2x/week     Low-impact movement in water for chronic pain
Long-term Management + Referral         Gentle Mobility Work     2    10              Daily                       Range of motion maintenance
          Home Exercise Program       

In [5]:
data_dict = {
    "patient_id"                      : "Unique patient identifier",
    "age"                             : "Patient age in years",
    "sex"                             : "Biological sex (Male / Female)",
    "race_ethnicity"                  : "Self-reported race/ethnicity category",
    "occupation_type"                 : "Occupational category",
    "work_related_injury"             : "Whether injury is work-related (Yes / No)",
    "region_census"                   : "US Census region",
    "pain_duration_weeks"             : "Duration of pain in weeks",
    "pain_duration_category"          : "Acute (<6 wks), Subacute (6-12 wks), Chronic (>12 wks)",
    "pain_severity_score_0_10"        : "Self-reported pain intensity (0 = none, 10 = worst)",
    "pain_severity_category"          : "Mild / Moderate / Severe categorization of severity score",
    "osw_baseline_percent_disability" : "Oswestry Disability Index total score (%)",
    "osw_baseline_disability_category": "Minimal / Moderate / Severe / Crippling / Bed-bound",
}

print("Data Dictionary\n" + "-" * 60)
for col, desc in data_dict.items():
    print(f"  {col:<44} {desc}")

Data Dictionary
------------------------------------------------------------
  patient_id                                   Unique patient identifier
  age                                          Patient age in years
  sex                                          Biological sex (Male / Female)
  race_ethnicity                               Self-reported race/ethnicity category
  occupation_type                              Occupational category
  work_related_injury                          Whether injury is work-related (Yes / No)
  region_census                                US Census region
  pain_duration_weeks                          Duration of pain in weeks
  pain_duration_category                       Acute (<6 wks), Subacute (6-12 wks), Chronic (>12 wks)
  pain_severity_score_0_10                     Self-reported pain intensity (0 = none, 10 = worst)
  pain_severity_category                       Mild / Moderate / Severe categorization of severity score
  osw_baseline_per

In [6]:
missing     = patients.isnull().sum()
missing_pct = (missing / len(patients) * 100).round(2)

completeness = pd.DataFrame({
    "missing_count": missing,
    "missing_pct"  : missing_pct
}).query("missing_count > 0")

if completeness.empty:
    print("No missing values detected.")
else:
    print("Columns with missing values:")
    print(completeness)

No missing values detected.


In [7]:
# Confirm every pathway in the patient dataset has a matching exercise record
patient_pathways  = set(patients["pain_duration_category"].unique())  # placeholder — update to pathway column after Notebook 03
exercise_pathways = set(exercises["pathway"].unique())

print("Pathways in exercises table:")
for p in sorted(exercise_pathways):
    print(f"  {p}")

Pathways in exercises table:
  Home Exercise Program
  In-Clinic Progression
  Long-term Management + Referral
  Standard Progression
  Urgent Referral


In [8]:
note = """
Dataset Notes
-------------
The patient dataset is entirely synthetic (n=500). It was generated to simulate
realistic lower back pain intake profiles for proof-of-concept development.

Limitations:
  - Statistical distributions may not reflect real clinical populations
  - Model performance metrics should not be extrapolated to real patients
  - Demographic proportions are illustrative, not representative
  - All findings require validation on real clinical data before use

The exercises dataset is a structured reference table mapping clinical
pathways to recommended exercise protocols. It is used for output
enrichment, not as a modeling input.
"""
print(note)


Dataset Notes
-------------
The patient dataset is entirely synthetic (n=500). It was generated to simulate
realistic lower back pain intake profiles for proof-of-concept development.

Limitations:
  - Statistical distributions may not reflect real clinical populations
  - Model performance metrics should not be extrapolated to real patients
  - Demographic proportions are illustrative, not representative
  - All findings require validation on real clinical data before use

The exercises dataset is a structured reference table mapping clinical
pathways to recommended exercise protocols. It is used for output
enrichment, not as a modeling input.

